In [0]:
%run ./adls_auth

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

LOG_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/pipeline_run_log"

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("activity_name", StringType(), False),
    StructField("status", StringType(), False),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("duration_seconds", LongType(), True),
    StructField("error_message", StringType(), True)
])

# Clear Spark catalog cache for this path
spark.catalog.clearCache()

# Check using Delta API directly rather than try/except
if not DeltaTable.isDeltaTable(spark, LOG_PATH):
    spark.createDataFrame([], schema=LOG_SCHEMA).write.format("delta").mode("overwrite").save(LOG_PATH)
    print("Created pipeline_run_log table successfully.")
else:
    print("pipeline_run_log already exists.")